# 02 - Modelagem com features temporais

Este notebook compara o desempenho dos modelos usando:

1. dataset base;
2. dataset com features temporais.

O objetivo é verificar se a engenharia temporal melhora o desempenho em relação à baseline de persistência.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parents[1]

BASE_DATA_PATH = PROJECT_ROOT / "data" / "final" / "conflict_country_year_base.csv"
TEMPORAL_DATA_PATH = PROJECT_ROOT / "data" / "final" / "conflict_country_year_temporal.csv"

df_base = pd.read_csv(BASE_DATA_PATH)
df_temporal = pd.read_csv(TEMPORAL_DATA_PATH)

print("Base dataset:", df_base.shape)
print("Temporal feature dataset:", df_temporal.shape)

Base dataset: (6737, 22)
Temporal feature dataset: (6737, 28)


In [2]:
TARGET_COLUMN = "target_conflict_next_year"
TRAIN_END_YEAR = 2016

BASE_FEATURE_COLUMNS = [
    "year",
    "state_based_conflict_exists",
    "state_based_dyad_count",
    "state_based_deaths_best",
    "intrastate_conflict_exists",
    "intrastate_deaths_best",
    "interstate_conflict_exists",
    "interstate_deaths_best",
    "non_state_conflict_exists",
    "non_state_dyad_count",
    "non_state_deaths_best",
    "one_sided_violence_exists",
    "one_sided_dyad_count",
    "one_sided_deaths_best",
    "cumulative_organized_violence_deaths_best",
    "organized_violence_exists",
]

TEMPORAL_FEATURE_COLUMNS = [
    "conflict_previous_year",
    "conflict_last_3_years_count",
    "conflict_last_5_years_count",
    "deaths_previous_year",
    "deaths_last_3_years_sum",
    "deaths_last_5_years_sum",
    "years_since_last_conflict",
]

EXPERIMENTS = {
    "dataset_base": {
        "data": df_base,
        "features": BASE_FEATURE_COLUMNS,
    },
    "dataset_temporal": {
        "data": df_temporal,
        "features": BASE_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS,
    },
}

for experiment_name, experiment in EXPERIMENTS.items():
    df = experiment["data"]
    features = experiment["features"]

    print("\n", experiment_name)
    print("Rows, columns:", df.shape)
    print("Number of features:", len(features))
    print("Target distribution:")
    print(df[TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))


 dataset_base
Rows, columns: (6737, 22)
Number of features: 16
Target distribution:
target_conflict_next_year
0    0.7051
1    0.2949
Name: proportion, dtype: float64

 dataset_temporal
Rows, columns: (6737, 28)
Number of features: 23
Target distribution:
target_conflict_next_year
0    0.7051
1    0.2949
Name: proportion, dtype: float64


In [3]:
def evaluate_predictions(experiment_name, model_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "experiment": experiment_name,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def build_models():
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        ),
        "Logistic Regression scaled": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            )),
        ]),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=4,
            class_weight="balanced",
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    }

In [4]:
results = []

# Persistence baseline
df_reference = df_base.copy()

train_mask = df_reference["year"] <= TRAIN_END_YEAR
test_mask = df_reference["year"] > TRAIN_END_YEAR

y_test_reference = df_reference.loc[test_mask, TARGET_COLUMN]
y_pred_persistence = df_reference.loc[test_mask, "organized_violence_exists"]

results.append(
    evaluate_predictions(
        "reference",
        "Persistence baseline",
        y_test_reference,
        y_pred_persistence,
    )
)

# Model experiments
for experiment_name, experiment in EXPERIMENTS.items():
    df = experiment["data"].copy()
    feature_columns = experiment["features"]

    train_mask = df["year"] <= TRAIN_END_YEAR
    test_mask = df["year"] > TRAIN_END_YEAR

    X_train = df.loc[train_mask, feature_columns]
    y_train = df.loc[train_mask, TARGET_COLUMN]

    X_test = df.loc[test_mask, feature_columns]
    y_test = df.loc[test_mask, TARGET_COLUMN]

    print("\nExperiment:", experiment_name)
    print("Train period:", df.loc[train_mask, "year"].min(), "-", df.loc[train_mask, "year"].max())
    print("Test period:", df.loc[test_mask, "year"].min(), "-", df.loc[test_mask, "year"].max())
    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)

    models = build_models()

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append(
            evaluate_predictions(
                experiment_name,
                model_name,
                y_test,
                y_pred,
            )
        )

results_df = pd.DataFrame(results)
results_df.round(4)


Experiment: dataset_base
Train period: 1989 - 2016
Test period: 2017 - 2023
Train shape: (5365, 16)
Test shape: (1372, 16)


C:\Users\enzo.going\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Experiment: dataset_temporal
Train period: 1989 - 2016
Test period: 2017 - 2023
Train shape: (5365, 23)
Test shape: (1372, 23)


C:\Users\enzo.going\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp
0,reference,Persistence baseline,0.9082,0.8571,0.8571,0.8571,868,63,63,378
1,dataset_base,Logistic Regression,0.9082,0.8604,0.8526,0.8565,870,61,65,376
2,dataset_base,Logistic Regression scaled,0.9082,0.8604,0.8526,0.8565,870,61,65,376
3,dataset_base,Decision Tree,0.9016,0.9527,0.7302,0.8267,915,16,119,322
4,dataset_base,Random Forest,0.9082,0.8571,0.8571,0.8571,868,63,63,378
5,dataset_temporal,Logistic Regression,0.9133,0.8546,0.8798,0.8670,865,66,53,388
6,dataset_temporal,Logistic Regression scaled,0.9162,0.8937,0.8390,0.8655,887,44,71,370
7,dataset_temporal,Decision Tree,0.8907,0.8198,0.8458,0.8326,849,82,68,373
8,dataset_temporal,Random Forest,0.9031,0.8277,0.8821,0.8540,850,81,52,389


In [5]:
results_sorted = results_df.sort_values(
    by=["f1_score", "recall", "precision"],
    ascending=False,
)

results_sorted.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp
5,dataset_temporal,Logistic Regression,0.9133,0.8546,0.8798,0.8670,865,66,53,388
6,dataset_temporal,Logistic Regression scaled,0.9162,0.8937,0.8390,0.8655,887,44,71,370
0,reference,Persistence baseline,0.9082,0.8571,0.8571,0.8571,868,63,63,378
4,dataset_base,Random Forest,0.9082,0.8571,0.8571,0.8571,868,63,63,378
1,dataset_base,Logistic Regression,0.9082,0.8604,0.8526,0.8565,870,61,65,376
2,dataset_base,Logistic Regression scaled,0.9082,0.8604,0.8526,0.8565,870,61,65,376
8,dataset_temporal,Random Forest,0.9031,0.8277,0.8821,0.8540,850,81,52,389
7,dataset_temporal,Decision Tree,0.8907,0.8198,0.8458,0.8326,849,82,68,373
3,dataset_base,Decision Tree,0.9016,0.9527,0.7302,0.8267,915,16,119,322


In [6]:
baseline_f1 = results_df.loc[
    results_df["model"] == "Persistence baseline",
    "f1_score"
].iloc[0]

best_by_experiment = (
    results_df
    .sort_values("f1_score", ascending=False)
    .groupby("experiment")
    .head(1)
    .reset_index(drop=True)
)

best_by_experiment["f1_difference_vs_persistence"] = (
    best_by_experiment["f1_score"] - baseline_f1
)

best_by_experiment.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp,f1_difference_vs_persistence
0,dataset_temporal,Logistic Regression,0.9133,0.8546,0.8798,0.8670,865,66,53,388,0.0099
1,reference,Persistence baseline,0.9082,0.8571,0.8571,0.8571,868,63,63,378,0.0000
2,dataset_base,Random Forest,0.9082,0.8571,0.8571,0.8571,868,63,63,378,0.0000


In [7]:
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_TABLES_DIR / "temporal_feature_model_results.csv"
best_results_path = OUTPUT_TABLES_DIR / "temporal_feature_best_results.csv"

results_df.to_csv(results_path, index=False)
best_by_experiment.to_csv(best_results_path, index=False)

print(f"Saved full results to: {results_path}")
print(f"Saved best results to: {best_results_path}")


Saved full results to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\temporal_feature_model_results.csv
Saved best results to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\temporal_feature_best_results.csv


## Interpretação esperada

A interpretação principal deve responder:

- As features temporais melhoraram algum modelo em relação ao dataset base?
- Algum modelo superou a baseline de persistência?
- Se não superou, isso reforça a hipótese de forte persistência temporal da violência organizada?
- Se superou, qual modelo melhorou e qual métrica foi mais impactada?

Para a qualificação, o resultado deve ser apresentado como achado metodológico, não como promessa de previsão determinística.